# Lesson 4.1 — EC2 AMIs, Launch Templates & EC2 Image Builder

**Level: Intermediate** | **Exam:** SOA-C03 Domain 3, Task 3.1 (Skills 3.1.1, 3.1.5)

## Learning Objectives
- Create and manage Amazon Machine Images (AMIs) for repeatable deployments
- Write EC2 Launch Templates with versioning
- Build automated, patched AMIs using EC2 Image Builder
- Understand the difference between AMI baking vs bootstrapping

_Reference: [EC2 Image Builder](https://docs.aws.amazon.com/imagebuilder/latest/userguide/) | [Launch Templates](https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/ec2-launch-templates.html)_

## AMI vs Launch Template vs User Data

| Approach | What it does | When to use |
|----------|-------------|-------------|
| **AMI baking** | Pre-install all software into image | Faster launch, consistent state |
| **User Data (bootstrapping)** | Run scripts at first launch | Flexible, updates without new AMI |
| **Combined** | Bake base + User Data for instance-specific config | Best of both worlds |

## EC2 Launch Templates

Launch Templates replace Launch Configurations (older). They support:
- **Versioning** — create new versions without replacing existing
- **Inheritance** — default version + overrides
- **Instance type flexibility** — specify multiple types for Spot pools
- Required by: ASG, EC2 Fleet, Spot Fleet

## EC2 Image Builder

Automates building, testing, and distributing AMIs on a schedule:
```
Pipeline:
Source AMI → Build Components → Test → Distribute to Regions → Share with Accounts
```
- **Components**: Scripts that install software (e.g., install Nginx, harden OS)
- **Image Recipe**: Source AMI + list of components + storage config
- **Infrastructure Config**: What EC2 instance type to build on
- **Distribution Config**: Where to share the finished AMI

In [ ]:
import boto3, json

ec2 = boto3.client('ec2', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# List your custom AMIs
response = ec2.describe_images(Owners=['self'])
amis = response['Images']

if amis:
    print(f'Custom AMIs ({len(amis)}):')
    for ami in sorted(amis, key=lambda x: x['CreationDate'], reverse=True):
        print(f'  {ami["ImageId"]}  {ami["Name"][:50]:50}  {ami["CreationDate"][:10]}')
else:
    print('No custom AMIs found in your account.')
    print('\nTo create an AMI from an existing instance:')
    print('ec2.create_image(')
    print('    InstanceId="i-1234567890abcdef0",')
    print('    Name="MyApp-v1.2.0-2024-01-01",')
    print('    NoReboot=False   # Ensures consistent filesystem state')
    print(')')
    print('\nBest practice AMI naming: <AppName>-<version>-<date>-<OS>')
    print('Example: myapp-2.1.0-20240101-al2023')

In [ ]:
# Create a Launch Template with production-ready settings

launch_template = {
    'LaunchTemplateName': 'cloudops-learning-webapp',
    'VersionDescription': 'v1 - Initial web application template',
    'LaunchTemplateData': {
        'ImageId': 'ami-0c02fb55956c7d316',  # Amazon Linux 2 (us-east-1) - verify latest
        'InstanceType': 't3.micro',
        'SecurityGroupIds': [],  # Would add actual SG IDs here
        'IamInstanceProfile': {'Name': 'EC2CloudWatchRole'},  # From Lesson 1.1
        'Monitoring': {'Enabled': True},  # Enable detailed monitoring
        'MetadataOptions': {
            'HttpEndpoint': 'enabled',
            'HttpTokens': 'required',      # IMDSv2 REQUIRED (security best practice)
            'HttpPutResponseHopLimit': 1
        },
        'TagSpecifications': [
            {
                'ResourceType': 'instance',
                'Tags': [
                    {'Key': 'Application', 'Value': 'cloudops-webapp'},
                    {'Key': 'Environment', 'Value': 'dev'},
                    {'Key': 'ManagedBy', 'Value': 'LaunchTemplate'}
                ]
            }
        ],
        'UserData': __import__('base64').b64encode(b'''#!/bin/bash
yum update -y
yum install -y nginx amazon-cloudwatch-agent
systemctl enable nginx
systemctl start nginx
echo "Instance $(curl -s http://169.254.169.254/latest/meta-data/instance-id) running!" > /var/www/html/index.html
''').decode()
    }
}

try:
    response = ec2.create_launch_template(**launch_template)
    lt_id = response['LaunchTemplate']['LaunchTemplateId']
    print(f'Created Launch Template: {lt_id}')
    print(f'Version: {response["LaunchTemplate"]["LatestVersionNumber"]}')
except ec2.exceptions.ClientError as e:
    if 'AlreadyExistsException' in str(e) or 'InvalidLaunchTemplateName' in str(e):
        lts = ec2.describe_launch_templates(LaunchTemplateNames=['cloudops-learning-webapp'])
        lt_id = lts['LaunchTemplates'][0]['LaunchTemplateId']
        print(f'Launch Template exists: {lt_id}')
    else:
        print(f'Error: {e}')

In [ ]:
# Create a new version of the launch template (e.g., upgraded instance type)
print('Launch Template Versioning:\n')
print('Versions let you update config without breaking existing ASGs:')
print('- v1: t3.micro (dev)')
print('- v2: t3.small (staging, more memory)')
print('- v3: t3.medium (prod, right-sized)')
print()
print('ASG can use: $Latest, $Default, or a specific version number')
print('Best practice: Set ASG to use $Default, update $Default only after testing')
print()
print('To create a new version:')
print('''ec2.create_launch_template_version(
    LaunchTemplateName='cloudops-learning-webapp',
    SourceVersion='1',
    VersionDescription='v2 - Upgraded to t3.small',
    LaunchTemplateData={'InstanceType': 't3.small'}
)''')

## Exercise

In [ ]:
# TODO: List all Launch Templates in your account and show:
# - Template name and ID
# - Number of versions
# - Default version number
# - Latest version number

import boto3
ec2 = boto3.client('ec2', region_name='us-east-1')

# YOUR CODE HERE
paginator = ec2.get_paginator('describe_launch_templates')
for page in paginator.paginate():
    for lt in page['LaunchTemplates']:
        # TODO: Print the required information
        pass

In [ ]:
# Cleanup
import boto3
ec2 = boto3.client('ec2', region_name='us-east-1')
try:
    ec2.delete_launch_template(LaunchTemplateName='cloudops-learning-webapp')
    print('Deleted Launch Template')
except Exception as e:
    print(f'Cleanup: {e}')

## Summary
- **AMI baking** = pre-installed software → faster, consistent launches
- **User Data** = bootstrapping at launch → flexible but slower, error-prone at scale
- **Launch Templates** replace Launch Configurations — use versioning for change management
- **IMDSv2** (`HttpTokens: required`) is a security best practice — prevents SSRF attacks
- **EC2 Image Builder** automates AMI creation, patching, testing, and distribution on a schedule

---
**Next:** [Lesson 4.2 — CloudFormation Advanced: StackSets, Nested Stacks, CDK](lesson-4.2-cloudformation-advanced.ipynb)